# Statistical Calibration and Conditional Power

Recompute the retained historical inference examples, 28 planning scenarios, 390,000 Student/Welch attempts, 100,000 known-bound attempts and 120,000 empirical-Bernstein attempts for the capped finite task. Full output coverage is listed in this folder's README. Historical methods and their negative calibration findings are distinct supporting diagnostics, not interchangeable with a later adopted inference method. These are synthetic statistical computations, not human-study evidence or theoretical prediction validation.

Requires a checkout, Python dependencies installed below, and Node.js for the pure scientific schedule extraction. No generated/private inputs are required.

In [ ]:
from pathlib import Path
import subprocess
import shutil
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'src').is_dir())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
if shutil.which('node') is None:
    raise RuntimeError('Node.js is required; install it separately before running this notebook')
subprocess.run(['node', '--version'], check=True)

## Settings and Scope

`smoke` uses the same seeds and 250-attempt batch zero of every cell; it is wiring only and overlaps full. `full` preserves every frozen cell, stage, policy, and criterion. Large computations are disabled. Outputs are written under ignored `results/`; use a fresh destination for each run.

The historical result retains MIXED/111 failures. The bounded and variance-adaptive sensitivities neither replace the primary method nor recover original-population targets under informative selection. Raw floating truths are not snapped to an ideal zero.

In [ ]:
import json
import shlex
RUN = False
MODE = 'smoke'  # 'full' preserves 390000 + 100000 + 120000 attempts
OUTPUT = Path('results/statistical_reproduction') / MODE
assert MODE in ('smoke', 'full')
configs = {
    'student_welch': 'configs/tutoring_calibration_v01.json',
    'bounded': 'configs/tutoring_bounded_robustness_v01.json',
    'variance_adaptive': 'configs/tutoring_identifiability_robustness_v01.json',
    'planning': 'configs/tutoring_power_sensitivity_v01.json',
}
for name, path in configs.items():
    spec = json.loads((ROOT / path).read_text())
    print(name, path)
    print(json.dumps(spec, indent=2))
print('Node.js:', shutil.which('node'))

In [ ]:
commands = []
for analysis in ('inference', 'planning', 'bounded', 'variance-adaptive'):
    commands.append([sys.executable, 'scripts/reproduce_tutoring_statistics.py', '--analysis', analysis, '--mode', MODE, '--output', str(OUTPUT / analysis)])
commands.append([sys.executable, 'scripts/calibrate_tutoring_methods.py', '--config', configs['student_welch'], '--mode', MODE, '--output', str(OUTPUT / 'student_welch')])
for command in commands:
    print(shlex.join(command))
if RUN:
    if shutil.which('node') is None:
        raise RuntimeError('Install Node.js before deterministic schedule preparation')
    for command in commands:
        subprocess.run(command, cwd=ROOT, check=True)

## Read Newly Computed Results

The reader below only opens outputs from this notebook's producer commands. It never accepts an external prepared context or historical result folder. Inspect all CSV/JSON diagnostics rather than treating successful process exit as all criteria passing. The bounded and variance-adaptive summaries retain method support, eligibility, paired comparisons, null rosters, selection diagnostics and failed criteria.

In [ ]:
if RUN:
    calibration = json.loads((ROOT / OUTPUT / 'student_welch/calibration_summary.json').read_text())
    planning = json.loads((ROOT / OUTPUT / 'planning/planning.json').read_text())
    print('Conditional planning scenarios:', len(planning['scenarios']))
    print('Inference examples:', [p.name for p in sorted((ROOT / OUTPUT / 'inference').glob('*.json'))])
    failures = json.loads((ROOT / OUTPUT / 'student_welch/criteria_readback.json').read_text())
    print('Student/Welch:', calibration['attempts'], 'attempts;', len(failures['failureRecords']), 'failed criteria;', failures['status'])
    for analysis in ('bounded', 'variance-adaptive'):
        result = json.loads((ROOT / OUTPUT / analysis / 'summary.json').read_text())
        print(analysis, result['attempts'], result['status'], 'failures:', len(result['criteriaFailures']))
    context = json.loads((ROOT / OUTPUT / 'variance-adaptive/prepared_context.json').read_text())
    print('Recomputed projection corners:', context['cornerCoefficients'])
    for name in ('C1_states.json', 'cohort_identification.json', 'source_slope_domain.json'):
        artifact = json.loads((ROOT / OUTPUT / 'variance-adaptive' / name).read_text())
        print(name, 'recomputed artifact entries:', len(artifact))
    for cell in context['cells']:
        truth = cell['truths']['selectedCore']
        print(cell['id'], 'raw nulls:', truth['rawNullRoster'], 'ideal nulls:', truth['idealNullRoster'])

## Limits

No universal Type-I guarantee, human behavior, final recruitment N, original-population MNAR repair, or new theory is inferred. Historical global-Holm and supporting alternatives retain their original policies. This notebook has no UI replay/normalization dependency and does not claim to reproduce prior report graphics.